In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [2]:
dataset = pd.read_csv('ai_jobs_salaries_clean.csv')
dataset.head(5)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,experience_level_label,employment_type_label,work_mode,salary_outlier_flag,role_family,isco_group_hint
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,Executive-level,Full-time,On-site,True,Analytics Manager,Information and communications technology serv...
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,Executive-level,Full-time,On-site,False,Analytics Manager,Information and communications technology serv...
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,Senior-level,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians"
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M,Senior-level,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians"
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M,Mid-level,Full-time,Remote,False,Other / Unclassified,NaN


In [3]:
dataset['High_Salary'] = (dataset['salary_in_usd'] >= 150000).astype(int)

leak_cols = [
    'High_Salary',
    'salary',
    'salary_currency',
    'salary_in_usd',
    'salary_outlier_flag',
]

In [4]:
print(dataset.isnull().sum())

work_year                     0
experience_level              0
employment_type               0
job_title                     0
salary                        0
salary_currency               0
salary_in_usd                 0
employee_residence            0
remote_ratio                  0
company_location              0
company_size                  0
experience_level_label        0
employment_type_label         0
work_mode                     0
salary_outlier_flag           0
role_family                   0
isco_group_hint           40540
High_Salary                   0
dtype: int64


In [5]:
dataset = dataset.dropna()

In [6]:
datatrain, datatest = train_test_split(dataset, test_size=0.2, shuffle=True)

In [7]:
X_train = datatrain.drop(columns=leak_cols)
y_train = datatrain['High_Salary']
X_test = datatest.drop(columns=leak_cols)
y_test = datatest['High_Salary']

In [8]:
X_train.head(5)

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,experience_level_label,employment_type_label,work_mode,role_family,isco_group_hint
59573,2024,SE,FT,Data Engineer,GB,0,GB,M,Senior-level,Full-time,On-site,Data Engineer,Software and applications developers
68156,2023,SE,FT,Data Engineer,US,0,US,M,Senior-level,Full-time,On-site,Data Engineer,Software and applications developers
63399,2024,SE,FT,Business Intelligence Analyst,US,100,US,M,Senior-level,Full-time,Remote,Data Analyst,Business services and administration managers
10079,2025,EN,FT,Data Scientist,US,100,US,M,Entry-level,Full-time,Remote,Data Scientist,"Mathematicians, actuaries and statisticians"
69749,2023,SE,FT,Data Scientist,US,0,US,M,Senior-level,Full-time,On-site,Data Scientist,"Mathematicians, actuaries and statisticians"


In [9]:
cols = ['experience_level', 'employment_type', 'job_title', 'employee_residence', 'company_location', 
        'company_size', 'experience_level_label', 'employment_type_label', 'work_mode', 'role_family', 'isco_group_hint']

In [10]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=2000, cat_features=cols, task_type='GPU')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

Learning rate set to 0.015173
0:	learn: 0.6882222	total: 43.5ms	remaining: 1m 26s
1:	learn: 0.6834636	total: 83.2ms	remaining: 1m 23s
2:	learn: 0.6790251	total: 122ms	remaining: 1m 21s
3:	learn: 0.6746339	total: 162ms	remaining: 1m 20s
4:	learn: 0.6703493	total: 204ms	remaining: 1m 21s
5:	learn: 0.6663763	total: 243ms	remaining: 1m 20s
6:	learn: 0.6623421	total: 284ms	remaining: 1m 20s
7:	learn: 0.6586296	total: 324ms	remaining: 1m 20s
8:	learn: 0.6548641	total: 363ms	remaining: 1m 20s
9:	learn: 0.6513536	total: 405ms	remaining: 1m 20s
10:	learn: 0.6479019	total: 443ms	remaining: 1m 20s
11:	learn: 0.6445176	total: 488ms	remaining: 1m 20s
12:	learn: 0.6413926	total: 527ms	remaining: 1m 20s
13:	learn: 0.6384560	total: 569ms	remaining: 1m 20s
14:	learn: 0.6355138	total: 608ms	remaining: 1m 20s
15:	learn: 0.6327273	total: 651ms	remaining: 1m 20s
16:	learn: 0.6300271	total: 690ms	remaining: 1m 20s
17:	learn: 0.6275423	total: 730ms	remaining: 1m 20s
18:	learn: 0.6251949	total: 771ms	remainin

In [12]:
score = accuracy_score(y_test, y_pred)
score

0.719203187250996